# Epithelial BBKNN Results Visualization

**Purpose**: Visualize filtered epithelial cells after BBKNN batch correction
- UMAP plots by batch, clustering, and cell_type_level_2/3/4
- Marker gene expression
- Batch effect assessment
- Cell composition analysis by different annotation levels

**Date**: 2026-01-09

## Configuration

In [ ]:
# ===== Configuration =====
INPUT_H5AD = "/home/h2048/data/py/0109/subset_qc_filter_bbknn_20260109_v1_1/adata_epithelial_filtered_bbknn.h5ad"
OUTDIR = "/home/h2048/data/py/0109/subset_qc_filter_bbknn_20260109_v1_1/figures"

# Key metadata columns
BATCH_KEY = "dataset"
LEIDEN_KEY = "leiden_bbknn"
ANNOTATION_LEVELS = ["cell_type_level_2", "cell_type_level_3", "cell_type_level_4"]

# Epithelial marker genes
EPITHELIAL_MARKERS = {
    'Basal': ['TP63', 'KRT5', 'KRT14', 'KRT15'],
    'Secretory': ['SCGB1A1', 'SCGB3A1', 'MUC5B', 'BPIFA1'],
    'Goblet': ['MUC5AC', 'MUC5B', 'TFF3', 'SPDEF'],
    'Ciliated': ['FOXJ1', 'CAPS', 'RSPH1', 'PIFO'],
    'Ionocyte': ['FOXI1', 'CFTR', 'ATP6V1B1'],
    'Tuft': ['POU2F3', 'TRPM5', 'GFI1B']
}

# Figure settings
FIG_DPI = 300
FIG_FORMAT = 'png'  # or 'pdf'
RANDOM_SEED = 0

## Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad

# Create output directory
os.makedirs(OUTDIR, exist_ok=True)

# scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=FIG_DPI, facecolor='white', frameon=False)
sc.settings.figdir = OUTDIR

print(f"Python: {sys.version.split()[0]}")
print(f"scanpy: {sc.__version__}")
print(f"anndata: {ad.__version__}")
print(f"\nInput: {INPUT_H5AD}")
print(f"Output: {OUTDIR}")

## 1. Load Data

In [ ]:
print("=" * 60)
print("Loading data...")
print("=" * 60)

adata = sc.read_h5ad(INPUT_H5AD)
print(f"\nData shape: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"\nAvailable obs columns:\n{list(adata.obs.columns)}")
print(f"\nAvailable obsm keys:\n{list(adata.obsm.keys())}")

# Check annotation levels
for level in ANNOTATION_LEVELS:
    if level in adata.obs.columns:
        n_valid = adata.obs[level].notna().sum()
        n_unique = adata.obs[level].nunique()
        print(f"\n{level}: {n_unique} unique values, {n_valid}/{adata.n_obs} valid cells")

## 2. Basic Statistics

In [ ]:
print("=" * 60)
print("Basic statistics")
print("=" * 60)

# Batch distribution
print("\n--- Batch Distribution ---")
batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
print(batch_counts)
print(f"\nTotal batches: {len(batch_counts)}")
print(f"Mean cells per batch: {batch_counts.mean():.1f}")
print(f"Median cells per batch: {batch_counts.median():.1f}")

# Leiden clustering
print(f"\n--- {LEIDEN_KEY} Distribution ---")
leiden_counts = adata.obs[LEIDEN_KEY].value_counts().sort_index()
print(leiden_counts)
print(f"\nTotal clusters: {len(leiden_counts)}")

### Cell Type Distribution by Levels

In [ ]:
for level in ANNOTATION_LEVELS:
    if level not in adata.obs.columns:
        print(f"\n⚠️  {level} not found in data")
        continue
    
    print(f"\n{'=' * 60}")
    print(f"--- {level} Distribution ---")
    print(f"{'=' * 60}")
    
    # Count including NA
    celltype_counts = adata.obs[level].value_counts(dropna=False)
    print(celltype_counts)
    
    # Percentage excluding NA
    celltype_counts_valid = adata.obs[level].value_counts(dropna=True)
    if len(celltype_counts_valid) > 0:
        celltype_pct = (celltype_counts_valid / celltype_counts_valid.sum() * 100).round(2)
        print("\nPercentage (excluding NA):")
        print(celltype_pct)

## 3. UMAP Visualization - Core Panels

### 3A. UMAP by Batch

In [ ]:
if 'X_umap' not in adata.obsm:
    print("⚠️  UMAP coordinates not found. Skipping UMAP plots.")
else:
    print("Plotting UMAP by batch...")
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata, 
        color=BATCH_KEY, 
        title='UMAP colored by Batch',
        frameon=False,
        show=False,
        ax=ax
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f'umap_batch.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    plt.close()

### 3B. UMAP by Leiden Clustering

In [ ]:
if 'X_umap' in adata.obsm and LEIDEN_KEY in adata.obs.columns:
    print("Plotting UMAP by Leiden clustering...")
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata, 
        color=LEIDEN_KEY, 
        title='UMAP colored by Leiden Clustering',
        legend_loc='on data',
        legend_fontsize=8,
        frameon=False,
        show=False,
        ax=ax
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f'umap_leiden.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    plt.close()

### 3C. UMAP by Cell Type Levels

In [ ]:
if 'X_umap' in adata.obsm:
    for level in ANNOTATION_LEVELS:
        if level not in adata.obs.columns:
            print(f"\n⚠️  {level} not found, skipping...")
            continue
        
        print(f"\nPlotting UMAP by {level}...")
        
        # Create a copy of the column with NA as string for plotting
        plot_col = f"{level}_plot"
        adata.obs[plot_col] = adata.obs[level].astype(str)
        adata.obs[plot_col] = adata.obs[plot_col].replace('nan', 'NA')
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sc.pl.umap(
            adata, 
            color=plot_col, 
            title=f'UMAP colored by {level}',
            frameon=False,
            show=False,
            ax=ax
        )
        plt.tight_layout()
        
        safe_name = level.replace('_', '-')
        plt.savefig(os.path.join(OUTDIR, f'umap_{safe_name}.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
        plt.show()
        plt.close()
        
        # Clean up temporary column
        adata.obs.drop(columns=[plot_col], inplace=True)

## 4. Marker Gene Expression on UMAP

In [ ]:
if 'X_umap' not in adata.obsm:
    print("⚠️  UMAP coordinates not found. Skipping marker gene plots.")
else:
    print("=" * 60)
    print("Marker gene expression on UMAP")
    print("=" * 60)
    
    # Check which markers are present
    all_markers = []
    for celltype, markers in EPITHELIAL_MARKERS.items():
        all_markers.extend(markers)
    
    available_markers = [g for g in all_markers if g in adata.var_names]
    missing_markers = [g for g in all_markers if g not in adata.var_names]
    
    print(f"\nAvailable markers: {len(available_markers)}/{len(all_markers)}")
    if missing_markers:
        print(f"Missing markers: {missing_markers}")

### Plot Markers by Cell Type

In [ ]:
if 'X_umap' in adata.obsm:
    for celltype, markers in EPITHELIAL_MARKERS.items():
        available = [g for g in markers if g in adata.var_names]
        if not available:
            print(f"\n⚠️  Skipping {celltype}: no markers available")
            continue
        
        print(f"\nPlotting {celltype} markers: {available}")
        
        # Determine grid size
        n_markers = len(available)
        ncols = min(4, n_markers)
        nrows = (n_markers + ncols - 1) // ncols
        
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
        if n_markers == 1:
            axes = [axes]
        else:
            axes = axes.flatten() if n_markers > 1 else [axes]
        
        for idx, gene in enumerate(available):
            sc.pl.umap(
                adata,
                color=gene,
                use_raw=True if adata.raw is not None else False,
                cmap='viridis',
                frameon=False,
                show=False,
                ax=axes[idx],
                title=f'{gene} ({celltype})'
            )
        
        # Hide extra subplots
        for idx in range(n_markers, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        safe_celltype = celltype.replace('/', '_').replace(' ', '_')
        plt.savefig(
            os.path.join(OUTDIR, f'umap_markers_{safe_celltype}.{FIG_FORMAT}'), 
            dpi=FIG_DPI, 
            bbox_inches='tight'
        )
        plt.show()
        plt.close()

## 5. Batch Effect Assessment

### 5A. Cells per Batch

In [ ]:
print("Plotting cells per batch...")
batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(max(10, len(batch_counts) * 0.4), 6))
batch_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Number of Cells per Batch', fontsize=14, pad=20)
ax.set_xlabel('Batch', fontsize=12)
ax.set_ylabel('Cell Count', fontsize=12)
ax.axhline(y=batch_counts.mean(), color='red', linestyle='--', label=f'Mean: {batch_counts.mean():.0f}')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, f'batch_cell_counts.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
plt.show()
plt.close()

### 5B. Cell Type Composition by Batch (for each level)

In [ ]:
for level in ANNOTATION_LEVELS:
    if level not in adata.obs.columns:
        print(f"\n⚠️  {level} not found, skipping composition analysis...")
        continue
    
    print(f"\nPlotting {level} composition by batch...")
    
    # Filter out NA values for composition analysis
    adata_valid = adata[adata.obs[level].notna()].copy()
    
    if adata_valid.n_obs == 0:
        print(f"  ⚠️  No valid cells for {level}")
        continue
    
    comp = pd.crosstab(
        adata_valid.obs[BATCH_KEY],
        adata_valid.obs[level],
        normalize='index'
    ) * 100
    
    fig, ax = plt.subplots(figsize=(max(12, len(comp.columns) * 0.8), max(8, len(comp.index) * 0.3)))
    sns.heatmap(
        comp,
        annot=True,
        fmt='.1f',
        cmap='YlOrRd',
        cbar_kws={'label': 'Percentage (%)'},
        ax=ax
    )
    ax.set_title(f'{level} Composition by Batch (%)', fontsize=14, pad=20)
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel('Batch', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    safe_name = level.replace('_', '-')
    plt.savefig(os.path.join(OUTDIR, f'batch_composition_{safe_name}.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    plt.close()
    
    # Save as CSV
    comp.to_csv(os.path.join(OUTDIR, f'batch_composition_{safe_name}.csv'))

## 6. Clustering Quality - Confusion Matrices

In [ ]:
if LEIDEN_KEY not in adata.obs.columns:
    print("⚠️  Leiden clustering not found, skipping confusion matrices...")
else:
    for level in ANNOTATION_LEVELS:
        if level not in adata.obs.columns:
            print(f"\n⚠️  {level} not found, skipping...")
            continue
        
        print(f"\nPlotting confusion matrix: Leiden vs {level}...")
        
        # Filter out NA values
        adata_valid = adata[adata.obs[level].notna()].copy()
        
        if adata_valid.n_obs == 0:
            print(f"  ⚠️  No valid cells for {level}")
            continue
        
        # Confusion matrix
        conf = pd.crosstab(
            adata_valid.obs[LEIDEN_KEY],
            adata_valid.obs[level]
        )
        
        # Plot heatmap
        fig, ax = plt.subplots(figsize=(max(10, len(conf.columns) * 0.8), max(8, len(conf.index) * 0.4)))
        sns.heatmap(
            conf,
            annot=True,
            fmt='d',
            cmap='Blues',
            cbar_kws={'label': 'Cell Count'},
            ax=ax
        )
        ax.set_title(f'Leiden Cluster vs {level}', fontsize=14, pad=20)
        ax.set_xlabel(level, fontsize=12)
        ax.set_ylabel('Leiden Cluster', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        safe_name = level.replace('_', '-')
        plt.savefig(os.path.join(OUTDIR, f'confusion_leiden_vs_{safe_name}.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
        plt.show()
        plt.close()
        
        # Save as CSV
        conf.to_csv(os.path.join(OUTDIR, f'confusion_leiden_vs_{safe_name}.csv'))

## 7. QC Metrics Visualization

In [ ]:
print("=" * 60)
print("QC metrics visualization")
print("=" * 60)

# Check which QC metrics are available
qc_cols = []
for col in ['n_genes', 'n_counts', 'percent_mito', 'percent_ribo']:
    # Check both standard names and common variants
    possible_names = [col, col.replace('_', '.'), 'n_genes_by_counts', 'total_counts', 'pct_counts_mt']
    for name in possible_names:
        if name in adata.obs.columns:
            qc_cols.append(name)
            break

if qc_cols:
    print(f"\nAvailable QC metrics: {qc_cols}")
    
    # Violin plot
    n_metrics = len(qc_cols)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5*n_metrics, 5))
    if n_metrics == 1:
        axes = [axes]
    
    for idx, metric in enumerate(qc_cols):
        sc.pl.violin(
            adata,
            keys=metric,
            groupby=BATCH_KEY,
            rotation=90,
            show=False,
            ax=axes[idx]
        )
        axes[idx].set_title(metric.replace('_', ' ').title())
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f'qc_metrics_by_batch.{FIG_FORMAT}'), dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print("\n⚠️  No QC metrics found in adata.obs")

## 8. Summary Statistics Export

In [ ]:
print("=" * 60)
print("Exporting summary statistics")
print("=" * 60)

summary_stats = {
    'total_cells': adata.n_obs,
    'total_genes': adata.n_vars,
    'n_batches': adata.obs[BATCH_KEY].nunique(),
    'n_leiden_clusters': adata.obs[LEIDEN_KEY].nunique() if LEIDEN_KEY in adata.obs.columns else 0,
}

# Add stats for each annotation level
for level in ANNOTATION_LEVELS:
    if level in adata.obs.columns:
        n_valid = adata.obs[level].notna().sum()
        n_unique = adata.obs[level].nunique()
        summary_stats[f'{level}_n_types'] = n_unique
        summary_stats[f'{level}_n_valid_cells'] = n_valid
        summary_stats[f'{level}_pct_valid'] = round(n_valid / adata.n_obs * 100, 2)

summary_df = pd.DataFrame([summary_stats])
summary_df.to_csv(os.path.join(OUTDIR, 'summary_statistics.csv'), index=False)

print("\nSummary statistics:")
print(summary_df.T)

## 9. Summary

In [ ]:
print("\n" + "=" * 60)
print("✅ Visualization complete!")
print("=" * 60)
print(f"\nAll figures saved to: {OUTDIR}")
print("\nGenerated files:")
for f in sorted(os.listdir(OUTDIR)):
    print(f"  - {f}")